In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False



In [2]:
from paicos.util import get_index_of_radial_range as RadialRange

In [3]:
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   305, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]

radius = 1e2*snap.length

# widths = np.array([2e2, 2e2, 2e2], dtype=float)
widths = np.ones(3)*2*radius.value

Gauss = snap.uq('G')
arepo_length = snap.length
density_unit = snap['0_Density'].uq
L_box = (arepo_length*(widths[0]))

In [4]:
all_pos = np.zeros((0,3))
all_mass = np.zeros(0)
all_hsml = np.zeros(0)
for parttype in range(0,3):
    idx_sphere = RadialRange(snap[f"{parttype}_Coordinates"],
                       center, 0*radius, radius)
    all_pos = np.concatenate((all_pos, snap[f"{parttype}_Coordinates"][idx_sphere]), axis=0)
    all_mass = np.concatenate((all_mass, snap[f"{parttype}_Masses"][idx_sphere]), axis=0)
    if (parttype==0):
        hsml = np.cbrt((snap[f"{parttype}_Volume"]) / (4.0 * np.pi / 3.0))
    else:
        hsml = snap[f'{parttype}_SubfindHsml']
    all_hsml = np.concatenate((all_hsml, hsml[idx_sphere]), axis=0)

Attempting to get derived variable: 0_Volume...	[DONE]

Attempting to get derived variable: 1_Masses...	[DONE]



In [5]:

pe = tc.PotentialEnergy(snap, center, widths, 
                        all_pos, all_mass, npix=64)

In [6]:
np.sum(pe.tile.mass_per_tile)

array(4770.63404857)

In [7]:
np.sum(pe.index)

1475048

In [8]:
np.sum(pe.tile.particles_per_tile)

array(1475048)

In [9]:
np.histogram(pe.tile.particles_per_tile.flatten())

(array([260793,   1137,    125,     45,     17,     14,      4,      1,
             1,      7]),
 array([  0. ,  41.8,  83.6, 125.4, 167.2, 209. , 250.8, 292.6, 334.4,
        376.2, 418. ]))

In [10]:
U_all = pe.compute_potential(all_hsml, angle=40)

In [11]:
U_all

<PaicosQuantity -0.01080153 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)>

In [12]:
from numba import njit, prange
import cupy as cp
from numba import cuda
import math


@cuda.jit()
def cuda_potential(masses, pos, hsml, potential):

    Np =  masses.shape[0]
    i = cuda.grid(1)

    if (i < Np):
        h = hsml[i]
        for j in range(Np):
            m1m2 = masses[i]*masses[j]
            if (i==j):
                potential[i] -= m1m2*(3./2.)/h
            else:
                r = math.sqrt((pos[i][0] - pos[j][0])**2 +
                             (pos[i][1] - pos[j][1])**2 +
                             (pos[i][2] - pos[j][2])**2)
                potential_tmp = m1m2/r
                if (r <= h):
                    potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                potential[i] -= potential_tmp
                


def direct_calculation_cuda(snap, positions, masses, smoothing_length):
    # snap_select = snap.select(index, parttype=0)

    Np = masses.shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        assert smoothing_length.shape == masses.shape
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    mass = cp.array(masses.value)
    pos = cp.array(positions.value)
    hsml = cp.array(smoothing_length.value)
    potential = cp.zeros_like(mass)

    threadsperblock = 256
    blocks_1d = (Np + (threadsperblock - 1)) // threadsperblock
    

    cuda_potential[blocks_1d, threadsperblock](mass, pos, hsml, potential)

    potential = cp.asnumpy(cp.sum(potential))
    
    G = 6.67430e-8 * snap.uq('cm^3 g^-1 s^-2')
    potential = 0.5 * G * potential * snap.mass**2 / snap.length
    return potential
    

In [13]:
direct_calculation_cuda(snap, all_pos, 
                            all_mass, all_hsml)

<PaicosQuantity -0.01080786 cm3 arepo_mass2 / (arepo_length g small_a small_h s2)>

In [ ]:
# U_bench = []
# runtime_bench = [] # in s
# for angle in angles:
#     U_tmp = []
#     result = %timeit -n1 -r5 -o U_tmp.append(pe.compute_potential(smoothing_length, angle=angle))
#     U_bench.append(U_tmp[0])
#     runtime_bench.append(result.average)

In [ ]:
# runtime_bench

In [ ]:
# U_direct = []
# # runtime_direct = [] # in s
# result = %timeit -n1 -r5 -o U_direct.append(direct_calculation_cuda(snap, snap['0_Coordinates'][pe.index], snap['0_Masses'][pe.index], smoothing_length[pe.index]))

# U_direct = U_direct[0]
# runtime_direct = result.average

In [ ]:
# U_direct

In [ ]:
# U_bench

In [ ]:
# relative_err = np.array([np.abs((U_bench[i]-U_direct)/U_direct).value for i in range(len(U_bench))])

In [ ]:


# fig, ax = plt.subplots(2, 1, sharex=True)

# ax[0].plot(angles, relative_err*100, ls='', marker='o', label='tiles')
# ax[1].plot(angles, runtime_bench, ls='', marker='o', label='tiles')
# ax[1].hlines(runtime_direct, xmin=-5, xmax=45, color='orange', ls='-', label='direct N^2')

# ax[0].text(20, 0.02, f'direct N^2: {U_direct.value:.6f}')
# ax[0].text(20, 0.015, f'theory: {U_theory.value:.6f}')

# ax[0].set_xlim(xmin=-5, xmax=45)

# ax[1].set_xlabel('angle')
# ax[0].set_ylabel('relative err [%]')
# ax[1].set_ylabel('timings [s]')
# ax[0].legend()
# ax[1].legend()

# fig.suptitle('Benchmark of gravitational potential for uniform sphere')
# plt.savefig('./uniform_sphere.pdf')

# plt.show()